# How to make a Polynomial Chaos Expansion (PCE)
---------
A PCE can create a surrogate model for a function, provided that function has uncertain inputs, such as this function:  
$$
f(\vec{x}, t; \vec{\xi})
$$
With independent variables $\vec{x}$, and $t$ (Note: the inclusion of $t$ means $f$ not need be stationary), and unknown parameters $\vec{\xi}$.

## PCE Definition
if the dimension of $\vec{\xi}$ (the *stochastic dimension*) $= n$, then $f$ can then be modelled by:
$$
f(\vec{x}, t; \vec{\xi}) \approx \sum_{k=0}^{\text{N\_basis}} c_k(\vec{x},t) \Psi_k(\vec{\xi})
$$

Let's break down each term:
#### $\text{\textbf{N\_basis}}$
$$
\text{N\_basis} = \frac{(n + p)!}{n! + p!}
$$
and $p$ is a hyper-parameter representing the *polynomial order*.

#### $\Psi_k(\vec{\xi})$
Is a collection of polynomials. In particular:
$$
\Psi_k(\vec{\xi}) = \prod_{j=1}^n \Psi_{\alpha_j^{(k)}}(\xi_j)
$$
Recall that $n$ is the stochastic dimension (i.e, the length of $\vec{\xi}$), so we are calling a different $\Psi_{\alpha_j^{(k)}}$ function for each element of $\vec{\xi}$.

$\Psi_{\alpha_j^{(j)}}$ is a member of a set of polynomials.
Polynomials are drawn from a specific *family* of polynomials depending on the distribution of $\vec{\xi}$:
* If $\vec{\xi}$ is *uniformly* distributed, we will draw from the [Legendre Polynomal Family](https://en.wikipedia.org/wiki/Legendre_polynomials).
* If $\vec{\xi}$ is distributed according to a *gaussian* distribution, we will use the [Hermite Polynomal Family](https://en.wikipedia.org/wiki/Hermite_polynomials).

Each member of these polynomial family can be represented by its highest power (eg. $P^{(n)}$ represents the $n^{\text{th}}$ order polynomial member of the family). For each order, there is exactly one polynomial in each family.

**Multi-indexing:** Therefore, the strange notation $\Psi_i{\alpha_j^{(k)}}$ means that we create a *set* of $j$ polynomals, whose polynomial order sums to k. Examples will help:

**TODO: ADD EXAMPLE**

We then solve each polynomial at location $\xi_j$, and multiply their products together to get $\Psi_k$

Most people don't create their own multi-index scheme. Instead, we typically use a library like python's itertools to create these:

#### $c_k$: Deterministic Coefficents
These coefficents bring us to the goal of a PCE.
We know, deterministically, each part of the general PCE equation except our $c_k$'s.
The goal of training the PCE is to find these coefficients.
While we outline the goal of training here, the actual mechanics of doing this will become much clearer in the next section.

To train the PCE, we evaluate our forward model with many realizations of $\vec{\xi}$, build a large number of $\Psi_k(\vec{\xi})'s$.
With that, we know what the model evaluations are equal to, we know what our $\Psi_k$ values are equal to, and we use one of two methods to solve for $c_k$.
Once we have trained values of $c_k$, we can calculate $\Psi_k$ with novel values of $\vec{\xi}$.

The two techniques for training $c_k$ are:
1. Spectral Projection: Cool method, very elegent, difficult to implement, and if not done right, scales very poorly. Will not cover here.
2. Least Squares: Not very elegant, but something YOU can actually implement. This is how we solve systems here.


## Vector Form of PCE 
The vector form of the PCE helps make this all much clearer, and paves a practical path forward to coding up a PCE. 
Here, we can see how we would use a *trained* PCE.
We give the PCE a novel $\vec{\xi}$, and ask it to tell us what the output of the forward model should be $\vec{y}$.

$$
\vec{y} = \bm{\Psi}(\vec{\xi}) \vec{c}
$$

### Training
----

Here we create $\text{N\_samples}$ evaluations of $f$, each with a different realization of $\vec{\xi}^{(i)}$.
We then populate the $\bm{\Psi}$ matrix based on the distribution of $\vec{\xi}$, and $\text{N\_basis}$ (which is a function of our hyper-parameter $p$).
Once $\bm{\Psi}$ is calculated, we solve for \vec{c} with a least squares method

$$
\begin{bmatrix}
f(\vec{x},t ; \vec{\xi}^{(0)}) \\
f(\vec{x},t ; \vec{\xi}^{(1)}) \\
... \\
... \\
f(\vec{x},t ; \vec{\xi}^{(\text{N\_samples} - 1)}) \\
\end{bmatrix}

=

\begin{bmatrix}
\Psi_0(\vec{\xi}^{(0)}) & \Psi_1(\vec{\xi}^{(0)}) & \cdots & \Psi_{\text{N\_basis}}(\vec{\xi}^{(0)}) \\
\Psi_0(\vec{\xi}^{(1)}) & \cdots & \cdots\\
\cdots \\
\Psi_0(\vec{\xi}^{(\text{N\_samples} - 1)}) & \Psi_1(\vec{\xi}^{(\text{N\_samples} - 1)}) & \cdots & \Psi_{\text{N\_basis} - 1}(\vec{\xi}^{(\text{N\_samples} - 1)}) \\
\end{bmatrix}


\begin{bmatrix}
c_0 \\
\cdots \\
c_{\text{N\_basis}}
\end{bmatrix}
$$